In [1]:
import joblib
import os
import pandas as pd
from functools import reduce
import itertools

pd.set_option('display.max_rows', 500)
pd.set_option('display.max_columns', 500)
pd.set_option('display.width', 100)

c:\Users\liuch\miniconda3\Lib\site-packages\pandas\core\arrays\masked.py:60: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.5' currently installed).
  from pandas.core import (
C:\Users\liuch\AppData\Local\Temp\ipykernel_21392\1063433330.py:3: DeprecationWarning: 
Pyarrow will become a required dependency of pandas in the next major release of pandas (pandas 3.0),
(to allow more performant data types, such as the Arrow string type, and better interoperability with other libraries)
but was not found to be installed on your system.
If this would cause problems for you,
please provide us feedback at https://github.com/pandas-dev/pandas/issues/54466
        
  import pandas as pd


In [2]:
def format_results(file_list, latex = False, cv = True):
    # Get data
    df_list = []
    for file in file_list:
        df = pd.concat(joblib.load(file), axis = 1)
        df_list.append(df)

    # Select and reorder columns
    results = reduce(lambda l, r: l.combine_first(r), df_list) * 100
    if cv == False:
        results = results.reindex(['pluginlg', 'pluginrf', 'splin', 'advrkhs', 'nysadvrkhs', 'rfreisz', 'nnet'], axis=1).dropna(how='all', axis=1)  # Use reindex as a quick hack in case some columns are missing
    else:
        results = results.reindex(['pluginlg_cfit', 'pluginrf_cfit', 'splin_cfit', 'advrkhs_cfit', 'nysadvrkhs_cfit', 'rfreisz_cfit', 'nnet_cfit'], axis=1).dropna(how='all', axis=1)

    # Reorder rows
    index0_present = results.index.get_level_values(0).unique().tolist()
    index0_order = [x for x in ['n=100', 'n=200', 'n=500', 'n=1000', 'n=2000'] if x in index0_present]
    index1_order = ['cov', 'bias', 'ci_length']  # Drop 'reg_rmse', 'reisz_rmse', 'rmse'
    order = pd.MultiIndex.from_product([index0_order, index1_order])
    results = results.reindex(order)
    
    # Formatting
    results = results.round(0).astype(int)
    if latex == True:
        results = print(results.to_latex(index = False))  # No row names
    
    return results

In [3]:
file_list_0 = [
   'gcv_results/synthetic_dgp0_100.joblib',
   'gcv_results/synthetic_dgp0_200.joblib',
   'gcv_results/synthetic_dgp0_500.joblib',
   'gcv_results/synthetic_dgp0_1000.joblib',
   'gcv_results/synthetic_dgp0_2000.joblib',
]

file_list_1 = [
   'gcv_results/synthetic_dgp1.joblib'
]

file_list_3 = [
   'gcv_results/synthetic_dgp3.joblib'
]

latex = False  # Produce latex code for tables?

In [4]:
# Nonlinear design {n = 1000, dim(W) = 10}, values multiplied by 100
# Sample splitting
results = format_results(file_list_3, latex = latex, cv = True)
results

pluginlg_cfit  pluginrf_cfit  splin_cfit  advrkhs_cfit  nysadvrkhs_cfit  \
n=1000 cov                   79             76          74            90               70   
       bias                 -13              1          -5            -5               -2   
       ci_length             55             29          33            53               32   

                  rfreisz_cfit  nnet_cfit  
n=1000 cov                  91         75  
       bias                  0         -6  
       ci_length            39         34

In [5]:
# Nonlinear design {n = 1000, dim(W) = 10}, values multiplied by 100
# No sample splitting
results = format_results(file_list_3, latex = latex, cv = False)
results

pluginlg  pluginrf  splin  advrkhs  nysadvrkhs  rfreisz  nnet
n=1000 cov              80        67     73       82          75       93    74
       bias            -11         1     -5       -2          -2        0    -6
       ci_length        49        25     31       37          30       38    34

In [6]:
# High dimensional design {n = 100, dim(W) = 100}, values multiplied by 100
# Sample splitting
results = format_results(file_list_1, latex = latex, cv = True)
results

pluginlg_cfit  pluginrf_cfit  splin_cfit  advrkhs_cfit  nysadvrkhs_cfit  \
n=100 cov                   92             91          93            89               88   
      bias                  45             16           8             6                6   
      ci_length            217             93          88            72               73   

                 rfreisz_cfit  nnet_cfit  
n=100 cov                  83          3  
      bias                 -7        -35  
      ci_length           127          6

In [7]:
# High dimensional design {n = 100, dim(W) = 100}, values multiplied by 100
# No sample splitting
results = format_results(file_list_1, latex = latex, cv = False)
results

pluginlg  pluginrf  splin  advrkhs  nysadvrkhs  rfreisz  nnet
n=100 cov              79        86     88       88          87       91     3
      bias              2         6      1        8           9       27   -33
      ci_length        64        71     75       76          78      185     3

In [8]:
# Simple design {dim(W) = 10}, values multiplied by 100
# Sample splitting
results = format_results(file_list_0, latex = latex, cv = True)
results

pluginlg_cfit  pluginrf_cfit  splin_cfit  advrkhs_cfit  nysadvrkhs_cfit  \
n=100  cov                   91             88          92            87               88   
       bias                  -4              2           1            -1               -1   
       ci_length            138             97         115            93               93   
n=200  cov                   98             94          96            96               91   
       bias                   1              1          -1            -1               -2   
       ci_length             83             66          76            76               60   
n=500  cov                   94             92          96            97               93   
       bias                   0              1           0             2                1   
       ci_length             47             42          46            55               41   
n=1000 cov                   91             86          94            95               88   
       bias                   0              3           2             3                3   
       ci_length             34             30          32            40               29   
n=2000 cov                   92             88          96            96               89   
       bias                   0              3           1             2                2   
       ci_length             23             21          22            28               21   

                  rfreisz_cfit  nnet_cfit  
n=100  cov                  89         50  
       bias                  6        -12  
       ci_length           105         35  
n=200  cov                  96         65  
       bias                  1         -7  
       ci_length            68         33  
n=500  cov                  92         81  
       bias                  1         -3  
       ci_length            43         34  
n=1000 cov                  89         92  
       bias                  4          1  
       ci_length            31         32  
n=2000 cov                  88         97  
       bias                  3          1  
       ci_length            21         23

In [9]:
# Simple design {dim(W) = 10}, values multiplied by 100
# No sample splitting
results = format_results(file_list_0, latex = latex, cv = False)
results

pluginlg  pluginrf  splin  advrkhs  nysadvrkhs  rfreisz  nnet
n=100  cov              84        83     85       83          83       86    46
       bias             -3        -1     -3       -2          -2        3   -12
       ci_length        91        75     92       83          85       86    32
n=200  cov              97        90     93       92          88       94    66
       bias              0         0     -2       -1          -3        0    -7
       ci_length        68        55     67       64          55       65    33
n=500  cov              88        87     91       93          89       88    77
       bias             -1         0     -1        0          -1        0    -4
       ci_length        42        35     41       42          38       40    33
n=1000 cov              86        80     87       89          85       87    88
       bias              0         1      0        1           2        1     0
       ci_length        27        24     26       27          25       26    27
n=2000 cov              89        86     92       90          89       87    92
       bias              0         1      0        1           1        1     0
       ci_length        20        18     19       20          19       19    20